# Lab 19: Built-In AI Gateway

Microsoft Foundry includes a **built-in AI Gateway** capability that provides centralized governance for your AI workloads. This lab demonstrates how to connect Foundry projects to the AI Gateway and manage them through the Azure portal.

> **Disclaimer:** The API interactions in this lab are for educational purposes only, intended to explain how these components work behind the scenes. They are **not** meant as templates for automation. Some features demonstrated here may currently only be available through the Azure portal and do not yet have automation templates or official SDK support.

## What is the Built-In AI Gateway?

The AI Gateway is a native Foundry feature that enables enterprise-grade governance without requiring external infrastructure:

| Capability | Description |
|------------|-------------|
| **Token Limits and Quotas** | Set fine-grained token limits and quotas for model deployments - per project or per agent |
| **Agent and Tool Lifecycle** | Inventory external agents and tools, manage their lifecycle with pause and resume actions |
| **Unified Traffic Management** | Unify endpoints, monitoring, and governance for all your AI workload traffic |
| **Security and Governance** | Configure advanced security and governance policies using the Azure portal |

## How This Differs from Lab 1A/1B

In earlier labs, we set up a **central landing zone with APIM** where agents connect to the hub and all traffic flows through APIM. The key difference in this lab:

| Aspect | Lab 1A/1B Pattern | Built-In AI Gateway (This Lab) |
|--------|-------------------|-------------------------------|
| **Connection** | Agent connects to hub | Entire **project** connects to hub |
| **Playground** | Not governed | Governed - policies apply even in Foundry Playground |
| **Agent Invocation** | Routed through APIM | Does **not** go through AI Gateway |
| **Use Case** | Full traffic control | Project-level governance with portal UI |

**Important:** Agent invocations do not flow through the Built-In AI Gateway. For full agent traffic governance, use the APIM pattern from Lab 1A and 1B. These patterns can be combined for comprehensive coverage.

## What You'll Learn

| Feature | Description |
|---------|-------------|
| Resource Links | Bidirectional links that make spokes visible in AI Gateway portal |
| Portal Experience | Manage token limits, quotas, and policies via GUI |
| Multi-Spoke Governance | Apply policies across all connected projects |
| **Managed Identity Auth** | Backend credentials for APIM to authenticate with AI Services |

## Key Configuration: Backend Credentials

For traffic to flow through APIM to Azure AI Services, the backend **must** include managed identity credentials:

```json
"credentials": {
    "managedIdentity": {
        "resource": "https://ai.azure.com/"
    }
}
```

Without this, APIM cannot authenticate with the AI Services endpoint.

## Prerequisites

- Complete **Lab 1A** (Landing Zone with APIM)
- Complete **Lab 1B** and/or **Lab 2A** (project spokes)

## Step 1: Load Configuration

In [1]:
import os
from pathlib import Path
from gateway_helpers import (
    load_env, az_rest, get_subscription_id, discover_spokes,
    connect_spoke_to_gateway, get_portal_url, list_ai_gateway_products,
    set_token_limit, get_product_token_limits,
    mask_resource_name, mask_subscription, mask_principal_id
)

# Load environment
ROOT = Path(__file__).parent.parent if '__file__' in dir() else Path.cwd().parent
load_env(str(ROOT / '.env'))
load_env(str(ROOT / '18-ai-gateway-governance/.env'))

# Configuration
APIM_URL = os.environ.get('APIM_URL', '')
APIM_NAME = APIM_URL.split('//')[1].split('.')[0] if APIM_URL else "foundry-apim-lagivk"
APIM_RG = "lab1a-foundry-lz-hub"
API_VERSION = "2024-05-01"

SUBSCRIPTION = get_subscription_id()
SPOKES = discover_spokes()

print(f"Landing Zone APIM: {mask_resource_name(APIM_NAME)}")
print(f"APIM Resource Group: {APIM_RG}")
print(f"\nDiscovered {len(SPOKES)} spokes:")
for s in SPOKES:
    print(f"  - {mask_resource_name(s['name'])} ({s['lab']})")

Landing Zone APIM: foundry-apim-******
APIM Resource Group: lab1a-foundry-lz-hub

Discovered 5 spokes:
  - foundry-hub-****** (Lab 1A (Landing Zone))
  - foundry-spoke-****** (Lab 1B)
  - contoso-team-****** (Lab 2A (Contoso Ltd))
  - fabrikam-team-****** (Lab 2A (Fabrikam Inc))
  - woodgrove-team-****** (Lab 2A (Woodgrove Bank))


## Step 2: Get APIM Identity

The APIM service uses a managed identity to authenticate with AI Services backends.

In [ ]:
apim = az_rest(
    "GET",
    f"https://management.azure.com/subscriptions/{SUBSCRIPTION}/resourceGroups/{APIM_RG}"
    f"/providers/Microsoft.ApiManagement/service/{APIM_NAME}?api-version={API_VERSION}"
)

if not apim or "identity" not in apim:
    raise Exception(f"Could not get APIM {APIM_NAME}. Check APIM_NAME and APIM_RG.")

APIM_PRINCIPAL_ID = apim["identity"]["principalId"]
print(f"APIM Principal ID: {mask_principal_id(APIM_PRINCIPAL_ID)}")

## Step 3: Connect Spokes to AI Gateway

For each spoke, this will:
1. Grant APIM access to the AI Services account (RBAC)
2. Create APIM backend **with managed identity credentials** (required for auth)
3. Create APIM API and operations
4. Discover projects and create APIM Products
5. Create resource links (Project -> APIM Product)

**Important:** The backend credentials configuration enables APIM to authenticate with Azure AI Services using its managed identity. Without this, traffic won't flow through APIM.

In [3]:
connected_projects = []

for spoke in SPOKES:
    projects = connect_spoke_to_gateway(
        spoke=spoke,
        subscription=SUBSCRIPTION,
        apim_name=APIM_NAME,
        apim_rg=APIM_RG,
        apim_principal_id=APIM_PRINCIPAL_ID,
        api_version=API_VERSION
    )
    connected_projects.extend(projects)

print(f"\n{'='*50}")
print(f"Connected {len(connected_projects)} project(s) to AI Gateway")
print(f"{'='*50}")
for p in connected_projects:
    if p.get("project"):
        print(f"  - {mask_resource_name(p['account'])}/{mask_resource_name(p['project'])} ({p['lab']})")
    else:
        print(f"  - {mask_resource_name(p['account'])} (account-level, {p['lab']})")


Processing: foundry-hub-****** (Lab 1A (Landing Zone))
  1. Setting up RBAC...
  2. Creating APIM backend...
  3. Creating APIM API...
  4. Creating API operations...
  5. Setting API policy...
  6. Creating account-level resource link...
  7. Discovering projects...
     No projects found - account-level link already created

Processing: foundry-spoke-****** (Lab 1B)
  1. Setting up RBAC...
  2. Creating APIM backend...
  3. Creating APIM API...
  4. Creating API operations...
  5. Setting API policy...
  6. Creating account-level resource link...
  7. Discovering projects...
     Found 1 project(s)
     Connecting: project-******
       OK - Product: foundry-spoke-h225xe-project-h225xe-ai
  Waiting 10s for APIM to stabilize...

Processing: contoso-team-****** (Lab 2A (Contoso Ltd))
  1. Setting up RBAC...
  2. Creating APIM backend...
  3. Creating APIM API...
  4. Creating API operations...
  5. Setting API policy...
  6. Creating account-level resource link...
  7. Discovering pro

## Step 4: Open AI Gateway Portal

Now that resource links are created, you can manage the AI Gateway in the portal:
- View all connected projects
- Set token limits per project
- Configure rate limiting policies
- Monitor usage and costs

In [4]:
print("AI Gateway Management Portal URLs")
print("=" * 50)

for p in connected_projects:
    if p.get("project"):
        portal_url = get_portal_url(SUBSCRIPTION, p["rg"], p["account"], p["project"])
        print(f"\n{mask_resource_name(p['account'])}/{mask_resource_name(p['project'])} ({p['lab']})")
        print(f"  (Portal URL available after running this cell)")
    else:
        print(f"\n{mask_resource_name(p['account'])} (account-level, no project portal URL)")

AI Gateway Management Portal URLs

foundry-hub-****** (account-level, no project portal URL)

foundry-spoke-******/project-****** (Lab 1B)
  (Portal URL available after running this cell)

contoso-team-******/inventory-ai (Lab 2A (Contoso Ltd))
  (Portal URL available after running this cell)

fabrikam-team-******/doc-****** (Lab 2A (Fabrikam Inc))
  (Portal URL available after running this cell)

woodgrove-team-******/risk-****** (Lab 2A (Woodgrove Bank))
  (Portal URL available after running this cell)


## Step 5: View Connected Products

List all APIM products that represent connected Foundry projects.

In [5]:
products = list_ai_gateway_products(SUBSCRIPTION, APIM_RG, APIM_NAME, API_VERSION)

print("AI Gateway Products")
print("=" * 50)

if products:
    print(f"\nFound {len(products)} product(s):\n")
    for p in products:
        print(f"  {mask_resource_name(p['displayName'])}")
        print(f"    ID: {mask_resource_name(p['name'])}")
        print(f"    State: {p['state']}\n")
else:
    print("\n  No AI Gateway products found")

AI Gateway Products

Found 4 product(s):

  contoso-team-rfcu2j / inventory-ai
    ID: contoso-team-rfcu2j-inventory-ai-ai
    State: published

  fabrikam-team-uw3pmr / doc-******
    ID: fabrikam-team-uw3pmr-doc-studio-ai
    State: published

  foundry-spoke-h225xe / project-******
    ID: foundry-spoke-h225xe-project-h225xe-ai
    State: published

  woodgrove-team-oxzv7n / risk-******
    ID: woodgrove-team-oxzv7n-risk-analysis-ai
    State: published



## Step 6: Set Token Limits Programmatically

When you set token limits in the portal UI, it adds `set-variable` elements to the **product policy**:

```xml
<policies>
    <inbound>
        <base />
        <set-variable name="tokenlimit-gpt-4o" value="10000" />
        <set-variable name="tokenquota-gpt-4o" value="100000|Daily" />
    </inbound>
    ...
</policies>
```

The API policy then reads these variables via `context.Variables` and applies `llm-token-limit`.

You can do the same programmatically:

In [6]:
# Set token limits on products (same as portal UI)
MODEL_NAME = os.environ.get('MODEL_NAME', 'gpt-4.1-mini')

if connected_projects:
    print(f"Setting token limits for model: {MODEL_NAME}")
    print("=" * 50)
    
    # Find the Landing Zone product first
    lz_project = next((p for p in connected_projects if "Landing Zone" in p.get("lab", "")), None)
    
    if lz_project and lz_project.get("product"):
        product_name = lz_project["product"]
        print(f"\n[Landing Zone] {mask_resource_name(product_name)}")
        
        # Set TPM limit for the landing zone model deployment
        success = set_token_limit(
            subscription=SUBSCRIPTION,
            apim_rg=APIM_RG,
            apim_name=APIM_NAME,
            product_name=product_name,
            deployment_name=MODEL_NAME,
            tokens_per_minute=50000,        # 50K TPM for landing zone
            quota=500000,                   # 500K token quota
            quota_period="Daily"
        )
        
        if success:
            print(f"  Token limit set: {MODEL_NAME}")
            print(f"  TPM: 50,000 | Quota: 500,000/day")
        else:
            print("  Failed to set token limit")
    else:
        print("\n[Landing Zone] No product found")
    
    # Set limits on other products (spokes)
    for p in connected_projects:
        if "Landing Zone" in p.get("lab", ""):
            continue  # Already handled above
            
        product_name = p.get("product")
        if not product_name:
            continue
            
        print(f"\n[{p['lab']}] {mask_resource_name(product_name)}")
        
        success = set_token_limit(
            subscription=SUBSCRIPTION,
            apim_rg=APIM_RG,
            apim_name=APIM_NAME,
            product_name=product_name,
            deployment_name=MODEL_NAME,
            tokens_per_minute=10000,        # 10K TPM for spokes
            quota=100000,                   # 100K token quota
            quota_period="Daily"
        )
        
        if success:
            print(f"  Token limit set: {MODEL_NAME}")
            print(f"  TPM: 10,000 | Quota: 100,000/day")
        else:
            print("  Failed to set token limit")
    
    # Show all configured limits
    print(f"\n{'='*50}")
    print("Configured Token Limits Summary")
    print(f"{'='*50}")
    for p in connected_projects:
        product_name = p.get("product")
        if product_name:
            limits = get_product_token_limits(SUBSCRIPTION, APIM_RG, APIM_NAME, product_name)
            label = mask_resource_name(p.get("project") or p.get("account"))
            print(f"\n{label} ({p['lab']}):")
            if limits:
                for deployment, config in limits.items():
                    tpm = config.get("tpm", "N/A")
                    quota = config.get("quota", "N/A")
                    period = config.get("period", "N/A")
                    print(f"  {deployment}: {tpm} TPM, {quota}/{period}")
            else:
                print("  No limits configured")
else:
    print("No connected projects - run Step 3 first")

Setting token limits for model: gpt-4.1-mini

[Landing Zone] No product found

[Lab 1B] foundry-spoke-h225xe-project-h225xe-ai
  Token limit set: gpt-4.1-mini
  TPM: 10,000 | Quota: 100,000/day

[Lab 2A (Contoso Ltd)] contoso-team-rfcu2j-inventory-ai-ai
  Token limit set: gpt-4.1-mini
  TPM: 10,000 | Quota: 100,000/day

[Lab 2A (Fabrikam Inc)] fabrikam-team-uw3pmr-doc-studio-ai
  Token limit set: gpt-4.1-mini
  TPM: 10,000 | Quota: 100,000/day

[Lab 2A (Woodgrove Bank)] woodgrove-team-oxzv7n-risk-analysis-ai
  Token limit set: gpt-4.1-mini
  TPM: 10,000 | Quota: 100,000/day

Configured Token Limits Summary

project-****** (Lab 1B):
  No limits configured

inventory-ai (Lab 2A (Contoso Ltd)):
  No limits configured

doc-****** (Lab 2A (Fabrikam Inc)):
  No limits configured

risk-****** (Lab 2A (Woodgrove Bank)):
  No limits configured


## Summary

You've connected Foundry projects to the AI Gateway and can now set token limits programmatically.

### Critical Configuration: Backend Credentials

For APIM to route traffic to Azure AI Services, backends **must** include managed identity credentials:

```python
# Backend config in gateway_helpers.py
{
    "properties": {
        "url": "https://{account}.services.ai.azure.com/",
        "protocol": "http",
        "credentials": {
            "managedIdentity": {
                "resource": "https://ai.azure.com/"
            }
        }
    }
}
```

### How Token Limits Work

| Layer | Policy | Purpose |
|-------|--------|---------|
| **Product** | `set-variable name="tokenlimit-{deployment}"` | Define limits per deployment |
| **API** | `llm-token-limit` | Enforce limits using `context.Variables` |

### Programmatic vs Portal

```python
# Equivalent to clicking "Set limit" in portal UI
set_token_limit(
    product_name="my-product",
    deployment_name="gpt-4o",
    tokens_per_minute=10000,
    quota=100000,
    quota_period="Daily"
)
```

### Policy Templates

See `policies/` folder:
- `ai-gateway.xml` - Full API policy with `llm-token-limit`
- `rate-limit.xml` - Request rate limiting

### Portal URL Format

```
https://ai.azure.com/nextgen/r/{encoded_sub},{rg},,{account},{project}/Operate/manage/gateway
```

### Troubleshooting

If traffic doesn't flow through APIM:
1. Check backend has `credentials.managedIdentity.resource = "https://ai.azure.com/"`
2. Verify APIM has Cognitive Services User role on the AI Services account
3. Ensure resource link exists from Project to APIM Product